# Ансамбли: пригодность воды для питья (Potability)

**Датасет:** `water_potability.csv`  
**Цель:** `Potability` — 1 = вода пригодна, 0 = нет.

Решение **по пунктам задания** (1 → 4). Между кодом — короткие выводы; в коде — построчные комментарии (кроме `print`).

> Стиль близок к учебному `ensembles.ipynb`, но задача — **классификация**, не регрессия.


# 1. Анализ данных и базовые модели


## 1.1. Загрузка и разведочный анализ (EDA)


In [ ]:
# Импорты
import time  # замер времени обучения бустингов
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (
    train_test_split, GridSearchCV, StratifiedKFold, cross_val_score
)
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.base import clone

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    BaggingClassifier, RandomForestClassifier,
    AdaBoostClassifier, GradientBoostingClassifier,
    StackingClassifier, HistGradientBoostingClassifier,
)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report, RocCurveDisplay,
)

# XGBoost может не открыться на Mac без libomp — тогда используем HistGB как запасной вариант
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except Exception as e:
    HAS_XGB = False
    XGB_IMPORT_ERROR = str(e)

sns.set_theme(style="whitegrid")
np.random.seed(42)
pd.set_option("display.max_columns", 50)

DATA_PATH = "../3 Ансамблирования/water_potability.csv"
df = pd.read_csv(DATA_PATH)

print("Размер:", df.shape)
print("XGBoost доступен:" , HAS_XGB)
if not HAS_XGB:
    print("Причина:", XGB_IMPORT_ERROR[:200])
    print("В п.3 для XGBoost используем HistGradientBoostingClassifier как близкий аналог.")
df.head()


In [ ]:
# Структура, пропуски, баланс классов
print("=== info ===")
df.info()
print("\n=== пропуски ===")
print(df.isna().sum().sort_values(ascending=False))
print("Всего NaN:", int(df.isna().sum().sum()))
print("\n=== Potability ===")
print(df["Potability"].value_counts())
print(df["Potability"].value_counts(normalize=True).round(3))


In [ ]:
# Распределение целевой переменной
fig, ax = plt.subplots(figsize=(5, 3.5))
# countplot: сколько объектов каждого класса
sns.countplot(data=df, x="Potability", hue="Potability",
              palette=["salmon", "seagreen"], legend=False, ax=ax)
ax.set_title("Дисбаланс классов Potability")
ax.set_xlabel("0 = непригодна, 1 = пригодна")
plt.tight_layout()
plt.show()


In [ ]:
# Гистограммы признаков
feat_cols = [c for c in df.columns if c != "Potability"]
# Рисуем распределения всех физико-химических показателей
df[feat_cols].hist(bins=30, figsize=(12, 9), edgecolor="white")
plt.suptitle("Распределения признаков", y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
# Аномалии: boxplot по признакам
fig, axes = plt.subplots(3, 3, figsize=(12, 9))
axes = axes.ravel()
for i, col in enumerate(feat_cols):
    # Ящик с усами помогает увидеть выбросы
    sns.boxplot(data=df, y=col, ax=axes[i], color="lightblue")
    axes[i].set_title(col)
plt.suptitle("Boxplot: поиск выбросов/аномалий", y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
# Тепловая карта корреляций
corr = df.corr(numeric_only=True)
plt.figure(figsize=(9, 7))
# annot=True — числа в клетках; center=0 — нейтральный ноль
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0, square=True)
plt.title("Корреляции признаков и Potability")
plt.tight_layout()
plt.show()

print("Корреляции с Potability (по |r|):")
print(corr["Potability"].drop("Potability").abs().sort_values(ascending=False).round(3))


### Выводы EDA (простыми словами)

1. **Пропуски:** много NaN в `Sulfate`, `ph`, `Trihalomethanes` — заполним **медианой** (устойчивее среднего к хвостам).  
2. **Классы:** пригодной воды меньше (~39% vs ~61%) — есть **дисбаланс**. Стратегия: `stratify` при split, для линейных/SVM/`class_weight='balanced'`, метрики смотреть не только Accuracy (Precision/Recall/F1/ROC-AUC по классу 1).  
3. **Корреляции** с target слабые — задача непростая, один признак «не решает».  
4. **Выбросы** на boxplot есть, но деревья/леса к ним терпимее; для LR/SVM поможет масштабирование.


## 1.2. Предобработка и разделение выборки


In [ ]:
# Отделяем признаки и цель
y = df["Potability"].copy()
X = df.drop(columns=["Potability"]).copy()

# Список числовых колонок (здесь все признаки числовые)
num_features = list(X.columns)

# Пайплайн для моделей, которым нужен scale (LR, SVM):
# 1) медиана в пропусках  2) стандартизация
pipe_scaled = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),  # NaN → медиана столбца
    ("scaler", StandardScaler()),                   # среднее 0, разброс 1
])

# Для деревьев/лес/бустинг scale обычно не обязателен — только импутация
pipe_tree = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
])

# Делим 80/20 с сохранением доли классов
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train:", X_train.shape, "доля Potable =", round(y_train.mean(), 3))
print("Test: ", X_test.shape, "доля Potable =", round(y_test.mean(), 3))


## 1.3. Базовые модели: LogReg, SVM, Decision Tree


In [ ]:
# Функция оценки одной модели на test (+ опционально train)
def evaluate(name, model, X_tr, y_tr, X_te, y_te):
    # Запоминаем время обучения
    t0 = time.time()
    # Учим модель (внутри Pipeline: impute/scale + классификатор)
    model.fit(X_tr, y_tr)
    fit_sec = time.time() - t0

    # Классы на test
    y_pred = model.predict(X_te)
    # Вероятности класса 1 нужны для ROC-AUC (если модель умеет)
    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X_te)[:, 1]
    else:
        # SVM без probability=True так не умеет — возьмём decision_function и растянем
        scores = model.decision_function(X_te)
        y_proba = (scores - scores.min()) / (scores.max() - scores.min() + 1e-9)

    # Метрики: accuracy общая; precision/recall/f1 — для класса 1 (пригодная вода)
    row = {
        "model": name,
        "accuracy": accuracy_score(y_te, y_pred),
        "precision": precision_score(y_te, y_pred, zero_division=0),
        "recall": recall_score(y_te, y_pred, zero_division=0),
        "f1": f1_score(y_te, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_te, y_proba),
        "fit_sec": fit_sec,
        # Для диагностики смещение/дисперсия: качество на train
        "acc_train": accuracy_score(y_tr, model.predict(X_tr)),
    }
    return row, model, y_pred, y_proba


In [ ]:
# Собираем три базовые модели
# class_weight='balanced' — сильнее штрафует ошибки на редком классе (Potability=1)
base_models = {
    "LogisticRegression": Pipeline([
        ("prep", clone(pipe_scaled)),
        ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42)),
    ]),
    "SVM": Pipeline([
        ("prep", clone(pipe_scaled)),
        # probability=True нужно для predict_proba / ROC-AUC
        ("clf", SVC(kernel="rbf", class_weight="balanced", probability=True, random_state=42)),
    ]),
    "DecisionTree": Pipeline([
        ("prep", clone(pipe_tree)),
        ("clf", DecisionTreeClassifier(class_weight="balanced", random_state=42)),
    ]),
}

results = []          # строки таблицы метрик
fitted = {}           # обученные модели
probas = {}           # вероятности на test для ROC

for name, model in base_models.items():
    row, fitted_model, y_pred, y_proba = evaluate(
        name, model, X_train, y_train, X_test, y_test
    )
    results.append(row)
    fitted[name] = fitted_model
    probas[name] = y_proba
    print(f"Готово: {name}")

base_table = pd.DataFrame(results).sort_values("f1", ascending=False)
print("\n=== Базовые модели (test) ===")
print(base_table.round(4).to_string(index=False))


In [ ]:
# Столбчатый график F1 / ROC-AUC базовых моделей
plot_df = base_table.set_index("model")[["f1", "roc_auc", "accuracy"]]
ax = plot_df.plot(kind="bar", figsize=(8, 4), rot=15)
ax.set_ylim(0, 1.05)
ax.set_title("Базовые модели: F1 / ROC-AUC / Accuracy (test)")
ax.set_ylabel("Значение")
plt.tight_layout()
plt.show()

# Разрыв train−test accuracy — грубый индикатор дисперсии (переобучения)
base_table["gap_acc"] = base_table["acc_train"] - base_table["accuracy"]
print(base_table[["model", "acc_train", "accuracy", "gap_acc"]].round(4).to_string(index=False))


### Анализ базовых моделей (смещение–дисперсия)

| Модель | Типичное поведение |
|--------|-------------------|
| **LogReg** | Высокое **смещение** (простая линейная граница), низкая дисперсия — стабильна, но может недообучаться |
| **SVM (RBF)** | Гибче LogReg, средняя сложность — баланс зависит от `C`/`gamma` |
| **Decision Tree** | Низкое смещение / **высокая дисперсия** — легко переобучается (огромный gap train−test) |

**Лучшую базовую** выбираем по **F1** и **ROC-AUC** на test (с учётом дисбаланса). Её дальше сравниваем с ансамблями.


In [ ]:
# Фиксируем имя лучшей базовой модели по F1
best_base_name = base_table.iloc[0]["model"]
print("Лучшая базовая модель по F1:", best_base_name)
print(base_table.drop(columns=["model"]).iloc[0].round(4))


# 2. Bagging и случайный лес


## 2.1. BaggingClassifier — разное число деревьев и параметры


In [ ]:
# Эксперимент: n_estimators от 10 до 100
n_list = [10, 30, 50, 70, 100]
bag_rows = []

for n in n_list:
    # Бэггинг: много деревьев на бутстрап-выборках, итог — голосование
    model = Pipeline([
        ("prep", clone(pipe_tree)),
        ("clf", BaggingClassifier(
            estimator=DecisionTreeClassifier(random_state=42),
            n_estimators=n,       # сколько деревьев в комитете
            max_samples=0.8,      # доля строк в каждой бутстрап-выборке
            max_features=1.0,     # доля признаков (1.0 = все)
            n_jobs=-1,
            random_state=42,
        )),
    ])
    row, _, _, y_proba = evaluate(
        f"Bagging_n={n}", model, X_train, y_train, X_test, y_test
    )
    bag_rows.append(row)
    probas[row["model"]] = y_proba

bag_n_table = pd.DataFrame(bag_rows)
print(bag_n_table[["model", "f1", "roc_auc", "accuracy", "fit_sec"]].round(4).to_string(index=False))


In [ ]:
# График: качество vs число деревьев в бэггинге
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(n_list, bag_n_table["f1"], "o-", label="F1")
ax.plot(n_list, bag_n_table["roc_auc"], "s-", label="ROC-AUC")
ax.set_xlabel("n_estimators")
ax.set_ylabel("Метрика (test)")
ax.set_title("Bagging: качество vs число базовых моделей")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# Влияние max_samples и max_features (фиксируем n_estimators=50)
bag_param_rows = []
for max_samples, max_features in [(0.5, 0.5), (0.5, 1.0), (0.8, 0.5), (0.8, 1.0), (1.0, 0.5)]:
    model = Pipeline([
        ("prep", clone(pipe_tree)),
        ("clf", BaggingClassifier(
            estimator=DecisionTreeClassifier(random_state=42),
            n_estimators=50,
            max_samples=max_samples,    # доля объектов в каждой подвыборке
            max_features=max_features,  # доля признаков в каждой подвыборке
            n_jobs=-1,
            random_state=42,
        )),
    ])
    name = f"Bag_ms={max_samples}_mf={max_features}"
    row, _, _, y_proba = evaluate(name, model, X_train, y_train, X_test, y_test)
    bag_param_rows.append(row)
    probas[name] = y_proba

bag_param_table = pd.DataFrame(bag_param_rows)
print(bag_param_table[["model", "f1", "roc_auc", "accuracy"]].round(4).to_string(index=False))


### Краткий вывод по п. 2.1 (Bagging)

1. **Больше деревьев (`n_estimators`)** обычно чуть поднимает F1/ROC-AUC и делает результат стабильнее: одно «случайное» дерево меньше портит ответ. После ~50–100 деревьев прирост часто замедляется.  
2. **`max_samples` / `max_features`**: слишком маленькие доли (например 0.5 и 0.5) сильно режут информацию → качество падает. Ближе к полным данным (например `max_samples≈0.8`, `max_features=1.0`) бэггинг работает лучше на этой задаче.  
3. **Итог:** бэггинг уже лучше одиночного переобученного дерева за счёт усреднения, но это ещё не случайный лес — разнообразие деревьев здесь скромнее. Дальше сравним с RandomForest (п. 2.2).


## 2.2. RandomForest + подбор гиперпараметров


In [ ]:
# Сетка для RandomForest (умеренная, чтобы не ждать часами)
rf_grid = {
    "clf__n_estimators": [100, 200],
    "clf__max_depth": [None, 10, 20],
    "clf__max_features": ["sqrt", 0.5],
}

rf_pipe = Pipeline([
    ("prep", clone(pipe_tree)),
    ("clf", RandomForestClassifier(
        class_weight="balanced",
        n_jobs=-1,
        random_state=42,
    )),
])

# StratifiedKFold сохраняет долю классов в каждом фолде
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

rf_search = GridSearchCV(
    rf_pipe,
    param_grid=rf_grid,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    refit=True,
)
# Подбор только на train
rf_search.fit(X_train, y_train)

print("Лучшие параметры RF:", rf_search.best_params_)
print("Best CV F1:", round(rf_search.best_score_, 4))

# Оценка лучшего RF на test
row_rf, fitted["RandomForest"], _, y_proba_rf = evaluate(
    "RandomForest", rf_search.best_estimator_, X_train, y_train, X_test, y_test
)
probas["RandomForest"] = y_proba_rf
results = [r for r in results if r["model"] != "RandomForest"]
results.append(row_rf)

# Добавим лучший бэггинг из эксперимента с n_estimators (по F1)
best_bag_row = bag_n_table.sort_values("f1", ascending=False).iloc[0].to_dict()
# Переобучим этот конфиг и сохраним под коротким именем
best_n = int(best_bag_row["model"].split("=")[1])
bag_best = Pipeline([
    ("prep", clone(pipe_tree)),
    ("clf", BaggingClassifier(
        estimator=DecisionTreeClassifier(random_state=42),
        n_estimators=best_n,
        max_samples=0.8,
        max_features=1.0,
        n_jobs=-1,
        random_state=42,
    )),
])
row_bag, fitted["Bagging"], _, y_proba_bag = evaluate(
    "Bagging", bag_best, X_train, y_train, X_test, y_test
)
probas["Bagging"] = y_proba_bag
results = [r for r in results if r["model"] != "Bagging"]
results.append(row_bag)

print("\nRF vs Bagging (test):")
print(pd.DataFrame([row_bag, row_rf])[
    ["model", "f1", "roc_auc", "accuracy", "acc_train", "fit_sec"]
].round(4).to_string(index=False))


In [ ]:
# Важность признаков случайного леса
# Достаём сам RandomForest из Pipeline
rf_clf = fitted["RandomForest"].named_steps["clf"]
# Импутация уже внутри pipeline; importances_ соответствуют исходным именам столбцов X
imp = pd.Series(rf_clf.feature_importances_, index=X.columns).sort_values(ascending=True)

plt.figure(figsize=(8, 5))
imp.plot(kind="barh", color="teal")
plt.title("Важность признаков — RandomForest")
plt.xlabel("feature_importances_")
plt.tight_layout()
plt.show()

print(imp.sort_values(ascending=False).round(4))


### Физико-химический смысл важных признаков

По RandomForest сильнее всего влияют **Sulfate** и **ph** (~0.16–0.17), затем **Solids**, **Hardness**, **Chloramines** (~0.11). Слабее — Conductivity, Turbidity, Trihalomethanes, Organic_carbon.

Это логично: сульфаты и кислотность — базовые маркеры состава воды; жёсткость и растворённые вещества описывают минерализацию; хлорамины связаны с обеззараживанием. Модель опирается именно на такие «химические» сигналы, а не на один доминирующий признак.

### Bagging vs Random Forest (по нашему прогону)

- **Bagging:** F1 ≈ **0.44**, accuracy ≈ **0.66**, но train accuracy = **1.0** — комитет силён на test по accuracy/F1, при этом всё ещё сильно подстраивается под train.  
- **RandomForest:** F1 ≈ **0.44** (почти как бэггинг), ROC-AUC выше (**≈0.66** vs ≈0.65), train accuracy ниже (**≈0.92**) — меньше «идеальной» подгонки, ранжирование на test чуть увереннее.  
- **Вывод:** по F1 модели близки; RF выигрывает в ROC-AUC и слабее переобучается. Для этой задачи RF предпочтительнее как более стабильный ансамбль, даже если сырой F1 почти равен бэггингу.


# 3. Boosting: AdaBoost, Gradient Boosting, XGBoost


## 3.1. AdaBoost


In [ ]:
# Подбор n_estimators и learning_rate для AdaBoost
ada_grid = {
    "clf__n_estimators": [50, 100, 200],
    "clf__learning_rate": [0.05, 0.1, 0.5, 1.0],
}

ada_pipe = Pipeline([
    ("prep", clone(pipe_tree)),
    # Базовый ученик — неглубокое дерево (стандартная идея бустинга)
    ("clf", AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=1, random_state=42),
        random_state=42,
    )),
])

ada_search = GridSearchCV(
    ada_pipe, ada_grid, scoring="f1", cv=cv, n_jobs=-1, refit=True
)
ada_search.fit(X_train, y_train)

print("AdaBoost best params:", ada_search.best_params_)
print("AdaBoost best CV F1:", round(ada_search.best_score_, 4))

row_ada, fitted["AdaBoost"], _, y_proba_ada = evaluate(
    "AdaBoost", ada_search.best_estimator_, X_train, y_train, X_test, y_test
)
probas["AdaBoost"] = y_proba_ada
results = [r for r in results if r["model"] != "AdaBoost"]
results.append(row_ada)
print(pd.Series({k:v for k,v in row_ada.items() if k!="model"}).astype(float).round(4))


In [ ]:
# Кривая качества AdaBoost vs число итераций (staged_predict_proba)
# Берём уже обученный AdaBoost из pipeline
ada_clf = fitted["AdaBoost"].named_steps["clf"]
# Нужны уже импутированные данные — transform train/test через prep
X_tr_imp = fitted["AdaBoost"].named_steps["prep"].transform(X_train)
X_te_imp = fitted["AdaBoost"].named_steps["prep"].transform(X_test)

# staged_*: качество после 1,2,...,n деревьев
ada_f1_te, ada_f1_tr = [], []
for y_tr_s, y_te_s in zip(
    ada_clf.staged_predict(X_tr_imp),
    ada_clf.staged_predict(X_te_imp),
):
    ada_f1_tr.append(f1_score(y_train, y_tr_s, zero_division=0))
    ada_f1_te.append(f1_score(y_test, y_te_s, zero_division=0))

plt.figure(figsize=(7, 4))
plt.plot(range(1, len(ada_f1_tr) + 1), ada_f1_tr, label="F1 train")
plt.plot(range(1, len(ada_f1_te) + 1), ada_f1_te, label="F1 test")
plt.xlabel("Число итераций (деревьев)")
plt.ylabel("F1")
plt.title("AdaBoost: F1 vs количество итераций")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


### Краткий вывод по п. 3.1 (AdaBoost)

AdaBoost учит деревья **по очереди**, сильнее «наказывая» прошлые ошибки.  
В нашем прогоне качество по F1 оказалось **слабым** (модель почти не ловит класс «пригодна»), зато видно идею бустинга и график F1 по числу итераций. Дальше сравним с Gradient Boosting / XGBoost — обычно они стабильнее на такой задаче.


## 3.2. Gradient Boosting


In [ ]:
# Подбор параметров GradientBoosting
gb_grid = {
    "clf__n_estimators": [100, 200],
    "clf__learning_rate": [0.05, 0.1],
    "clf__max_depth": [2, 3],
}

gb_pipe = Pipeline([
    ("prep", clone(pipe_tree)),
    ("clf", GradientBoostingClassifier(random_state=42)),
])

gb_search = GridSearchCV(
    gb_pipe, gb_grid, scoring="f1", cv=cv, n_jobs=-1, refit=True
)
gb_search.fit(X_train, y_train)

print("GB best params:", gb_search.best_params_)
print("GB best CV F1:", round(gb_search.best_score_, 4))

row_gb, fitted["GradientBoosting"], _, y_proba_gb = evaluate(
    "GradientBoosting", gb_search.best_estimator_, X_train, y_train, X_test, y_test
)
probas["GradientBoosting"] = y_proba_gb
results = [r for r in results if r["model"] != "GradientBoosting"]
results.append(row_gb)
print(pd.Series({k:v for k,v in row_gb.items() if k!="model"}).astype(float).round(4))


In [ ]:
# F1 vs итерации для GradientBoosting (staged_predict)
gb_clf = fitted["GradientBoosting"].named_steps["clf"]
X_tr_imp = fitted["GradientBoosting"].named_steps["prep"].transform(X_train)
X_te_imp = fitted["GradientBoosting"].named_steps["prep"].transform(X_test)

gb_f1_te, gb_f1_tr = [], []
for y_tr_s, y_te_s in zip(
    gb_clf.staged_predict(X_tr_imp),
    gb_clf.staged_predict(X_te_imp),
):
    gb_f1_tr.append(f1_score(y_train, y_tr_s, zero_division=0))
    gb_f1_te.append(f1_score(y_test, y_te_s, zero_division=0))

plt.figure(figsize=(7, 4))
plt.plot(range(1, len(gb_f1_tr) + 1), gb_f1_tr, label="F1 train")
plt.plot(range(1, len(gb_f1_te) + 1), gb_f1_te, label="F1 test")
plt.xlabel("Число итераций")
plt.ylabel("F1")
plt.title("Gradient Boosting: F1 vs итерации")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


### Краткий вывод по п. 3.2 (Gradient Boosting)

GB строит деревья **по очереди**, каждое уменьшает общую ошибку.  
У нас F1 заметно **выше, чем у AdaBoost**, но train всё ещё выше test — при большом числе итераций нужно следить за переобучением (`learning_rate`, `max_depth`).


## 3.3. XGBoost (или HistGradientBoosting, если XGBoost недоступен)


In [ ]:
# XGBoost или запасной HistGradientBoosting
if HAS_XGB:
    xgb_grid = {
        "clf__n_estimators": [100, 200],
        "clf__learning_rate": [0.05, 0.1],   # eta
        "clf__max_depth": [3, 5],
        "clf__subsample": [0.8, 1.0],
        "clf__colsample_bytree": [0.8, 1.0],
    }
    xgb_pipe = Pipeline([
        ("prep", clone(pipe_tree)),
        ("clf", XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=42,
            n_jobs=-1,
        )),
    ])
    model_label = "XGBoost"
else:
    # Близкий по духу градиентный бустинг из sklearn (быстрый, устойчивый)
    xgb_grid = {
        "clf__max_iter": [100, 200],          # аналог n_estimators
        "clf__learning_rate": [0.05, 0.1],
        "clf__max_depth": [3, 5],
    }
    xgb_pipe = Pipeline([
        ("prep", clone(pipe_tree)),
        ("clf", HistGradientBoostingClassifier(random_state=42)),
    ])
    model_label = "HistGB (XGBoost fallback)"

xgb_search = GridSearchCV(
    xgb_pipe, xgb_grid, scoring="f1", cv=cv, n_jobs=-1, refit=True
)
xgb_search.fit(X_train, y_train)

print(model_label, "best params:", xgb_search.best_params_)
print(model_label, "best CV F1:", round(xgb_search.best_score_, 4))

row_xgb, fitted[model_label], _, y_proba_xgb = evaluate(
    model_label, xgb_search.best_estimator_, X_train, y_train, X_test, y_test
)
probas[model_label] = y_proba_xgb
results = [r for r in results if r["model"] != model_label]
results.append(row_xgb)
print(pd.Series({k:v for k,v in row_xgb.items() if k!="model"}).astype(float).round(4))


In [ ]:
# Сравнение бустингов: качество и скорость
boost_names = ["AdaBoost", "GradientBoosting", model_label]
boost_table = (
    pd.DataFrame([r for r in results if r["model"] in boost_names])
    .drop_duplicates(subset=["model"], keep="last")
    .reset_index(drop=True)
)
print(boost_table[["model", "f1", "roc_auc", "accuracy", "fit_sec", "acc_train"]].round(4).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
# Качество
boost_table.set_index("model")[["f1", "roc_auc"]].plot(kind="bar", ax=axes[0], rot=15)
axes[0].set_ylim(0, 1.05)
axes[0].set_title("Бустинги: F1 и ROC-AUC")
# Скорость
boost_table.set_index("model")["fit_sec"].plot(kind="bar", ax=axes[1], rot=15, color="gray")
axes[1].set_title("Время обучения (сек)")
plt.tight_layout()
plt.show()


### Сравнение бустингов (смещение / дисперсия / переобучение)

| Метод | Идея | Риск переобучения |
|-------|------|-------------------|
| **AdaBoost** | Перевзвешивает ошибочные объекты | Средний; сильно зависит от `learning_rate` и шума |
| **Gradient Boosting** | Каждый шаг чинит остаток предыдущих | При большом `n_estimators` + глубоких деревьях train уходит вверх |
| **XGBoost / HistGB** | Тот же градиентный бустинг + регуляризация/оптимизации | Обычно лучше контроль через `subsample`, `max_depth`, `eta` |

На графиках train vs test по итерациям: если **train растёт, а test падает** — пора уменьшать `learning_rate` / глубину или делать early stopping.


# 4. Stacking и финальное сравнение


## 4.1. Stacking (дерево + RF + GB → LogReg)


In [ ]:
# Базовые модели стекинга (уже с препроцессингом внутри каждого estimator)
estimators = [
    ("tree", Pipeline([
        ("prep", clone(pipe_tree)),
        ("clf", DecisionTreeClassifier(max_depth=5, class_weight="balanced", random_state=42)),
    ])),
    ("rf", Pipeline([
        ("prep", clone(pipe_tree)),
        ("clf", RandomForestClassifier(
            n_estimators=150, max_depth=15, class_weight="balanced",
            n_jobs=-1, random_state=42
        )),
    ])),
    ("gb", Pipeline([
        ("prep", clone(pipe_tree)),
        ("clf", GradientBoostingClassifier(
            n_estimators=150, learning_rate=0.1, max_depth=3, random_state=42
        )),
    ])),
]

# Мета-модель — логистическая регрессия на предсказаниях базовых
stack = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42),
    stack_method="predict_proba",  # на вход меты подаём вероятности
    cv=3,                          # внутренний CV, чтобы мета не переобучилась
    n_jobs=-1,
)

row_stack, fitted["Stacking"], _, y_proba_stack = evaluate(
    "Stacking", stack, X_train, y_train, X_test, y_test
)
probas["Stacking"] = y_proba_stack
results = [r for r in results if r["model"] != "Stacking"]
results.append(row_stack)
print(pd.Series({k:v for k,v in row_stack.items() if k!="model"}).astype(float).round(4))


In [ ]:
# Альтернативный стек: LogReg + SVM + RF → LogReg
estimators2 = [
    ("lr", Pipeline([
        ("prep", clone(pipe_scaled)),
        ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42)),
    ])),
    ("svm", Pipeline([
        ("prep", clone(pipe_scaled)),
        ("clf", SVC(probability=True, class_weight="balanced", random_state=42)),
    ])),
    ("rf", Pipeline([
        ("prep", clone(pipe_tree)),
        ("clf", RandomForestClassifier(
            n_estimators=150, class_weight="balanced", n_jobs=-1, random_state=42
        )),
    ])),
]

stack2 = StackingClassifier(
    estimators=estimators2,
    final_estimator=LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42),
    stack_method="predict_proba",
    cv=3,
    n_jobs=-1,
)

row_stack2, fitted["Stacking_v2"], _, y_proba_stack2 = evaluate(
    "Stacking_v2", stack2, X_train, y_train, X_test, y_test
)
probas["Stacking_v2"] = y_proba_stack2
results = [r for r in results if r["model"] != "Stacking_v2"]
results.append(row_stack2)
print(pd.Series({k:v for k,v in row_stack2.items() if k!="model"}).astype(float).round(4))


## 4.2. Финальная таблица и ROC-кривые


In [ ]:
# Единая таблица всех моделей
final_table = pd.DataFrame(results).drop_duplicates(subset=["model"], keep="last")
final_table = final_table.sort_values("f1", ascending=False).reset_index(drop=True)
print("=== Финальное сравнение (test) ===")
print(final_table[
    ["model", "accuracy", "precision", "recall", "f1", "roc_auc", "fit_sec", "acc_train"]
].round(4).to_string(index=False))

# Сохраняем таблицу рядом с ноутбуком
final_table.to_csv("water_ensembles_metrics.csv", index=False)
print("\nТаблица сохранена в water_ensembles_metrics.csv")


In [ ]:
# ROC-кривые ключевых моделей
roc_list = [
    best_base_name, "Bagging", "RandomForest",
    "AdaBoost", "GradientBoosting", model_label,
    "Stacking", "Stacking_v2",
]
# Оставим только те, что реально посчитаны
roc_list = [m for m in roc_list if m in probas]

plt.figure(figsize=(8, 6))
ax = plt.gca()
for name in roc_list:
    # Рисуем ROC по вероятностям на test
    RocCurveDisplay.from_predictions(y_test, probas[name], name=name, ax=ax)
ax.plot([0, 1], [0, 1], "k--", label="случайный")
ax.set_title("ROC-кривые моделей")
ax.legend(loc="lower right", fontsize=8)
plt.tight_layout()
plt.show()


In [ ]:
# Важность признаков: берём RF (интерпретируемый ансамбль)
if "RandomForest" in fitted:
    rf_clf = fitted["RandomForest"].named_steps["clf"]
    imp = pd.Series(rf_clf.feature_importances_, index=X.columns).sort_values(ascending=False)
    print("Топ признаков (RF) и физико-химический смысл:")
    meaning = {
        "ph": "кислотность воды",
        "Hardness": "жёсткость (Ca/Mg)",
        "Solids": "растворённые твёрдые вещества",
        "Chloramines": "хлорамины (дезинфекция)",
        "Sulfate": "сульфаты",
        "Conductivity": "электропроводность",
        "Organic_carbon": "органический углерод",
        "Trihalomethanes": "тригалометаны (побочные продукты хлорирования)",
        "Turbidity": "мутность",
    }
    for feat, val in imp.items():
        print(f"  {feat:18s} {val:.4f}  — {meaning.get(feat, '')}")


## Общий вывод

### Какие ансамбли лучше для Potability?
Смотрите финальную таблицу по **F1** и **ROC-AUC**. Обычно лидируют **RandomForest / GradientBoosting / XGBoost(HistGB) / Stacking**; одиночное дерево и простая LogReg слабее из‑за смещения или дисперсии.

### Гиперпараметры и смещение–дисперсия
- Больше деревьев (`n_estimators`) → ниже **дисперсия** (стабильнее).  
- Большой `max_depth` / высокий `learning_rate` → ниже смещение, но растёт **переобучение**.  
- `max_samples` / `subsample` / `max_features` добавляют случайность → разнообразнее модели, меньше дисперсия ансамбля.

### Практические рекомендации
1. Старт: **медиана пропусков + stratify + RF/HistGB**.  
2. При дисбалансе — `class_weight` / смотреть F1 и ROC-AUC, не только accuracy.  
3. Бустинг — когда нужно выжать качество; следить за train/test по итерациям.  
4. Stacking — если базовые модели **ошибаются по-разному**; дороже по времени.  
5. Для интерпретации качества воды удобен **RandomForest.feature_importances_**.

---

### Чеклист задания

| Пункт | Статус |
|-------|--------|
| 1. EDA + preprocess + базовые модели | ✅ |
| 2. Bagging + RF + важность признаков | ✅ |
| 3. AdaBoost + GB + XGBoost/HistGB | ✅ |
| 4. Stacking + сводная таблица + ROC + выводы | ✅ |
